### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="early_stage_diabetes_risk_prediction",
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5VG8H",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/529/early+stage+diabetes+risk+prediction+dataset.zip && unzip early+stage+diabetes+risk+prediction+dataset.zip && rm early+stage+diabetes+risk+prediction+dataset.zip && mkdir -p local-data-warehouse/early_stage_diabetes_risk_prediction && mv diabetes_data_upload.csv local-data-warehouse/early_stage_diabetes_risk_prediction/
""",
    # References
    academic_reference_bibtex="""@inproceedings{islam2019likelihood,
  title={Likelihood prediction of diabetes at early stage using data mining techniques},
  author={Islam, MM Faniqul and Ferdousi, Rahatara and Rahman, Sadikur and Bushra, Humayra Yasmin},
  booktitle={Computer Vision and Machine Intelligence in Medical Image Analysis: International Symposium, ISCMM 2019},
  pages={113--125},
  year={2019},
  organization={Springer}
}
""",
    academic_reference_bibtex_key="islam2019likelihood",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use the data as is from UCI.

- Note, the dataset contains a lot of naturally occurring duplicates (50%). We drop them to avoid too extreme duplicated-based data leakage biasing the evaluation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "diabetes_data_upload.csv")
print("Loaded data shape:", df.shape)

as_cat_type = list(df)
as_cat_type.remove("Age")
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop_duplicates()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (520, 17)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 251
Columns: 17
Use sampling: False (sample size: 251)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Age', 'Gender', 'Polyuria', 'sudden weight loss', 'Polydipsia', 'Polyphagia', 'Genital thrush', 'visual blurring', 'weakness', 'Itching']
Rows remaining as candidates after top-10 filter: 16 (of 251)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,55,Male,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,No,Yes,No,No,No,Positive
1,57,Male,Yes,Yes,No,Yes,Yes,Yes,No,No,No,Yes,Yes,No,No,No,Positive
2,61,Female,Yes,No,No,No,Yes,No,No,No,Yes,No,No,No,Yes,No,Positive
3,65,Female,Yes,Yes,No,Yes,Yes,No,No,Yes,No,No,Yes,Yes,No,No,Positive
4,55,Female,Yes,No,Yes,No,No,Yes,Yes,Yes,No,Yes,Yes,No,No,No,Positive


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Gender,category,0.0,0.0,2.0,"Male, Female"
1,Polyuria,category,0.0,0.0,2.0,"Yes, No"
2,Polydipsia,category,0.0,0.0,2.0,"No, Yes"
3,sudden weight loss,category,0.0,0.0,2.0,"No, Yes"
4,weakness,category,0.0,0.0,2.0,"Yes, No"
5,Polyphagia,category,0.0,0.0,2.0,"No, Yes"
6,Genital thrush,category,0.0,0.0,2.0,"No, Yes"
7,visual blurring,category,0.0,0.0,2.0,"No, Yes"
8,Itching,category,0.0,0.0,2.0,"Yes, No"
9,Irritability,category,0.0,0.0,2.0,"No, Yes"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,251.0,48.864542,12.526036,16.0,90.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column             rank                        
Alopecia           1           No    161  64.14
                   2          Yes     90  35.86
Gender             1         Male    160  63.75
                   2       Female     91  36.25
Genital thrush     1           No    184  73.31
                   2          Yes     67  26.69
Irritability       1           No    180  71.71
                   2          Yes     71  28.29
Itching            1          Yes    127  50.60
                   2           No    124  49.40
Obesity            1           No    207  82.47
                   2          Yes     44  17.53
Polydipsia         1           No    127  50.60
                   2          Yes    124  49.40
Polyphagia         1           No    134  53.39
                   2          Yes    117  46.61
Polyuria           1          Yes    132  52.59
                   2           No    119  47.41
class              1     Positive    173  68.92
                   2     Negative     78  31.08
delayed healing    1           No    126  50.20
                   2          Yes    125  49.80
muscle stiffness   1           No    153  60.96
                   2          Yes     98  39.04
partial paresis    1           No    139  55.38
                   2          Yes    112  44.62
sudden weight loss 1           No    147  58.57
                   2          Yes    104  41.43
visual blurring    1           No    140  55.78
                   2          Yes    111  44.22
weakness           1          Yes    159  63.35
                   2           No     92  36.65

In [8]:
# Target Distribution
target_df

,count,pct
class,,
Positive,173,68.92
Negative,78,31.08


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to early_stage_diabetes_risk_prediction/019d5dc0-b128-7f63-a2fa-7d2d5b2146e8
019d5dc0-b128-7f63-a2fa-7d2d5b2146e8
9d93119a10378fd42fe3d3329abc70223ac482e62ae90b8824da6e80c61d9953
